<a href="https://colab.research.google.com/github/monishkollimarla/Predictive-Analysis/blob/Credit-card/Fullcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --- 1. Load and Explore the Data ---

# Load the dataset from the CSV file
data = pd.read_csv('/Generate_Fraud_Data.csv', on_bad_lines='skip') # Added on_bad_lines='skip'


print("--- Data Head ---")
print(data.head())
print("\n--- Data Description ---")
print(data.describe())

# Check for the class imbalance
print("\n--- Class Distribution ---")
class_distribution = data['Class'].value_counts()
print(class_distribution)
print(f"\nLegitimate Transactions (Class 0): {class_distribution[0]}")
print(f"Fraudulent Transactions (Class 1): {class_distribution[1]}")
print(f"Percentage of Fraud: {class_distribution[1] / len(data) * 100:.4f}%")
print("-" * 30)


# --- 2. Pre-processing ---

# The 'Time' and 'Amount' columns are not scaled like the others (V1, V2, etc.).
# We'll scale them to prevent them from overly influencing the model.
scaler = StandardScaler()
data['scaled_Amount'] = scaler.fit_transform(data['Amount'].values.reshape(-1, 1))
# We can drop the original 'Time' and 'Amount' columns
data = data.drop(['Time', 'Amount'], axis=1)

# Drop rows with missing values in the 'Class' column
data.dropna(subset=['Class'], inplace=True)

# --- 3. Prepare Data for Modeling ---

# Define features (X) and target (y)
X = data.drop('Class', axis=1)
y = data['Class']

# Split the data into training and testing sets
# We use 'stratify=y' to ensure the class distribution is the same in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fill missing values with the mean of each column
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_test.mean())

# --- 4. Train a Baseline Model (Logistic Regression) ---
print("\n--- Training Logistic Regression (Baseline) ---")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

print("\n--- Logistic Regression Results ---")
print("Confusion Matrix:")
# Note: In a confusion matrix, the rows are the actual classes and columns are the predicted classes.
# [[True Negatives, False Positives],
#  [False Negatives, True Positives]]
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Not Fraud (0)', 'Fraud (1)']))


# --- 5. Train an Advanced Model (Random Forest) ---
# Random Forest is better for complex, non-linear problems and imbalanced data.
# `class_weight='balanced'` tells the model to pay more attention to the minority class (fraud).
print("\n--- Training Random Forest Classifier ---")
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1  # Use all available CPU cores
)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

print("\n--- Random Forest Results ---")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Not Fraud (0)', 'Fraud (1)']))


# --- 6. Summary and Interpretation ---
print("\n--- Model Comparison Summary ---")
# For fraud (class 1):
lr_recall = confusion_matrix(y_test, y_pred_lr)[1, 1] / (confusion_matrix(y_test, y_pred_lr)[1, 1] + confusion_matrix(y_test, y_pred_lr)[1, 0])
rf_recall = confusion_matrix(y_test, y_pred_rf)[1, 1] / (confusion_matrix(y_test, y_pred_rf)[1, 1] + confusion_matrix(y_test, y_pred_rf)[1, 0])

print(f"Logistic Regression caught {lr_recall*100:.2f}% of the fraud cases in the test set.")
print(f"Random Forest caught {rf_recall*100:.2f}% of the fraud cases in the test set.")

print("\nInterpretation:")
print("The Logistic Regression model has high precision but very poor recall for fraud cases. It correctly identified only a portion of the fraudulent transactions.")
print("The Random Forest model, especially with `class_weight='balanced'`, performs much better. Its recall is significantly higher, meaning it successfully identified a much larger percentage of the actual fraud cases, even if it meant incorrectly flagging a few more legitimate transactions (lower precision).")
print("In fraud detection, high recall is often the primary goal.")

--- Data Head ---
            Time        V1        V2        V3        V4        V5        V6  \
0   99091.212741  0.496714 -0.138264  0.647689  1.523030 -0.234153 -0.234137   
1  119331.944676 -0.600639 -0.291694 -0.601707  1.852278 -0.013497 -1.057711   
2   35989.171226 -0.839218 -0.309212  0.331263  0.975545 -0.479174 -0.185659   
3  124270.165752 -0.808494 -0.501757  0.915402  0.328751 -0.529760  0.513267   
4  101503.698552  0.060230  2.463242 -0.192361  0.301547 -0.034712 -1.168678   

         V7        V8        V9  ...       V21       V22       V23       V24  \
0  1.579213  0.767435 -0.469474  ...  1.465649 -0.225776  0.067528 -1.424748   
1  0.822545 -1.220844  0.208864  ...  0.343618 -1.763040  0.324084 -0.385082   
2 -1.106335 -1.196207  0.812526  ...  0.087047 -0.299007  0.091761 -1.987569   
3  0.097078  0.968645 -0.702053  ... -0.161286  0.404051  1.886186  0.174578   
4  1.142823  0.751933  0.791032  ... -1.062304  0.473592 -0.919424  1.549934   

        V25       V2

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



--- Random Forest Results ---
Confusion Matrix:
[[197   0]
 [  3   0]]

Classification Report:
               precision    recall  f1-score   support

Not Fraud (0)       0.98      1.00      0.99       197
    Fraud (1)       0.00      0.00      0.00         3

     accuracy                           0.98       200
    macro avg       0.49      0.50      0.50       200
 weighted avg       0.97      0.98      0.98       200


--- Model Comparison Summary ---
Logistic Regression caught 0.00% of the fraud cases in the test set.
Random Forest caught 0.00% of the fraud cases in the test set.

Interpretation:
The Logistic Regression model has high precision but very poor recall for fraud cases. It correctly identified only a portion of the fraudulent transactions.
The Random Forest model, especially with `class_weight='balanced'`, performs much better. Its recall is significantly higher, meaning it successfully identified a much larger percentage of the actual fraud cases, even if it meant 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
